# STORM: Perspective-Driven Research for Long-Form Writing

## What is STORM?

**STORM** (Synthesis of Topic Outlines through Retrieval and Multi-perspective question asking) is a
multi-agent orchestration pattern for producing well-researched, long-form articles on a topic the
writer has little prior expertise in. Instead of asking a single LLM to "write an article about X" in
one shot — which tends to produce shallow, generic prose — STORM breaks the task into four
collaborating stages, each handled by a differently-instructed agent role:

1. **Perspective generation** — an LLM proposes several distinct "editor" personas, each representing a
   different lens on the topic (e.g. an economist, an urban planner, a labor-market analyst).
2. **Simulated expert interviews** — for every persona, a Persona-Editor agent interviews an Expert
   agent across several question/answer turns, surfacing facts and insights specific to that lens.
3. **Outline synthesis** — an editor-in-chief agent merges all interview findings into a single,
   non-redundant, ordered article outline.
4. **Section drafting** — a writer agent drafts each outlined section grounded in the relevant research
   notes, and the sections are assembled into the final article.

This notebook implements a simplified version of the pattern (inspired by the community
`all-agentic-architectures` "STORM" example), swapping STORM's original live web-retrieval step for an
LLM playing a "knowledgeable expert" role. The interview → outline → draft skeleton generalizes directly
to a version wired up to a real search tool (e.g. `TavilySearch`, used elsewhere in this repo) — only
the Expert agent's tool access would need to change.

### High-level workflow

```
topic
  |
  v
[1] Perspective Generation  ->  N editor personas
  |
  v
[2] Simulated Interviews    ->  persona x (question -> answer) x k turns  ->  research notes per persona
  |
  v
[3] Outline Generation      ->  ordered sections synthesized from all research notes
  |
  v
[4] Section Drafting        ->  one agent call per section, grounded in relevant notes
  |
  v
final long-form article
```

### When to use it

- Long-form content (reports, wiki-style articles, briefings) on topics that benefit from multiple
  stakeholder viewpoints.
- Tasks where a single-shot "write about X" prompt produces shallow or one-sided output.
- Research workflows where you want an auditable trail of *which* question surfaced *which* fact before
  it made it into the final draft.

### Strengths / Weaknesses

**Strengths**
- Diversifies research angles automatically instead of relying on the writer to already know what
  questions matter.
- Each stage's intermediate output (perspectives, transcripts, outline) is inspectable — good for
  debugging and trust.
- Naturally parallelizable: every perspective's interview is independent of the others.

**Weaknesses**
- No built-in fact-checking: the "expert" here is only as reliable as the underlying LLM's knowledge,
  unless grounded with a real search tool.
- Cost scales linearly with perspectives x interview turns x sections.
- Persona quality bounds output quality — vague or overlapping personas produce redundant research.


In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ============ IMPORTS & LLM SETUP ============
from typing import Dict, List

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from pydantic import BaseModel, Field

from helpers import get_llm

llm = get_llm()

## Step 1: Perspective Generation

**What we are going to do:** give the LLM a topic and ask it to propose a handful of distinct "editor"
personas — each one a different professional lens that will drive its own line of research later. We
use a Pydantic schema with `with_structured_output` so we get a clean, typed list back instead of having
to parse free text.

In [ ]:
# ============ PERSPECTIVE SCHEMA & GENERATION ============
TOPIC = "The impact of remote work on urban commercial real estate"
NUM_PERSPECTIVES = 4


class Perspective(BaseModel):
    persona: str = Field(description="A short role title for this editor, e.g. 'Urban Planner'")
    description: str = Field(
        description="1-2 sentences on what this persona cares about and what angle they will research"
    )


class PerspectiveList(BaseModel):
    perspectives: List[Perspective] = Field(
        description="Distinct, non-overlapping research perspectives on the topic"
    )


perspective_llm = llm.with_structured_output(PerspectiveList)

perspective_prompt = [
    SystemMessage(
        content=(
            "You are organizing a panel of editors to research a long-form article. "
            f"Propose {NUM_PERSPECTIVES} distinct editor personas, each bringing a genuinely different "
            "professional lens to the topic below. Avoid overlapping angles."
        )
    ),
    HumanMessage(content=f"Topic: {TOPIC}"),
]

perspectives = perspective_llm.invoke(perspective_prompt).perspectives

In [ ]:
# ============ PRINT GENERATED PERSPECTIVES ============
print(f"Topic: {TOPIC}\n")
print(f"Generated {len(perspectives)} perspectives:\n")
for p in perspectives:
    print(f"- {p.persona}: {p.description}")

**Discussion of the output:** each persona should read as a genuinely different angle on the same
topic (e.g. an economist worrying about aggregate demand vs. a real-estate investor worrying about asset
valuations) rather than four reworded versions of the same generalist. If personas overlap heavily,
tightening the system prompt (e.g. explicitly listing disciplines to draw from) usually fixes it.

## Step 2: Simulated Expert Interviews

**What we are going to do:** for each persona, run a short simulated conversation between two agent
roles built from the *same* underlying LLM but with different system prompts:

- **Persona-Editor agent** — stays in character as the persona, asking one focused, probing question at
  a time, building on the previous answer.
- **Expert agent** — answers as a knowledgeable subject-matter expert, concisely and factually.

After a fixed number of turns, a third call summarizes the transcript into a short list of concrete
research facts/insights (again via structured output), which is what later stages will actually consume.

In [ ]:
# ============ INTERVIEW SCHEMA & SIMULATION ============
NUM_INTERVIEW_TURNS = 3


class ResearchFindings(BaseModel):
    facts: List[str] = Field(
        description="Concise, factual bullet points uncovered in the interview, relevant to this persona's angle"
    )


findings_llm = llm.with_structured_output(ResearchFindings)


def conduct_interview(topic: str, perspective: Perspective, num_turns: int = NUM_INTERVIEW_TURNS):
    """Simulate a multi-turn Q&A between a Persona-Editor agent and an Expert agent.

    Returns (transcript, findings) where transcript is a list of {"question", "answer"} dicts.
    """
    editor_system = SystemMessage(
        content=(
            f"You are {perspective.persona}, an editor researching the topic: \"{topic}\".\n"
            f"Your focus: {perspective.description}\n"
            "Ask ONE focused, probing question at a time to a subject-matter expert to gather "
            "information relevant to your focus. Build on prior answers. Output ONLY the question."
        )
    )
    expert_system = SystemMessage(
        content=(
            f"You are a knowledgeable subject-matter expert on \"{topic}\". "
            "Answer the editor's questions factually and concisely (2-4 sentences). "
            "If you are not certain of a fact, give your best reasoned estimate and say it is an estimate."
        )
    )

    editor_history: List = [editor_system]
    expert_history: List = [expert_system]
    transcript = []

    for turn in range(num_turns):
        ask = "Ask your first question." if turn == 0 else "Ask your next question."
        question = llm.invoke(editor_history + [HumanMessage(content=ask)]).content.strip()

        answer = llm.invoke(expert_history + [HumanMessage(content=question)]).content.strip()

        transcript.append({"question": question, "answer": answer})
        editor_history += [AIMessage(content=question), HumanMessage(content=f"Expert answered: {answer}")]
        expert_history += [AIMessage(content=answer)]

    transcript_text = "\n".join(f"Q: {t['question']}\nA: {t['answer']}" for t in transcript)
    findings = findings_llm.invoke(
        [
            SystemMessage(
                content=(
                    "Extract concise, factual bullet points from this interview transcript, relevant to "
                    f"the perspective: {perspective.persona} — {perspective.description}"
                )
            ),
            HumanMessage(content=transcript_text),
        ]
    )
    return transcript, findings

In [ ]:
# ============ RUN INTERVIEWS FOR ALL PERSPECTIVES ============
transcripts: Dict[str, list] = {}
research_notes: Dict[str, ResearchFindings] = {}

for perspective in perspectives:
    transcript, findings = conduct_interview(TOPIC, perspective)
    transcripts[perspective.persona] = transcript
    research_notes[perspective.persona] = findings

In [ ]:
# ============ PRINT A SAMPLE INTERVIEW TRANSCRIPT ============
sample_persona = perspectives[0]
print(f"Sample interview -- persona: {sample_persona.persona}\n")
for i, turn in enumerate(transcripts[sample_persona.persona], start=1):
    print(f"Turn {i}")
    print(f"  Q: {turn['question']}")
    print(f"  A: {turn['answer']}\n")

print("Extracted research findings:")
for fact in research_notes[sample_persona.persona].facts:
    print(f"- {fact}")

**Discussion of the output:** the Persona-Editor's questions should visibly track its persona's stated
focus and get more specific turn over turn (a real interview, not four independent restatements of the
same opening question). The extracted `facts` list is the actual payload that survives into the outline
and drafting stages — the raw transcript is kept mainly for auditability.

## Step 3: Outline Generation

**What we are going to do:** hand every perspective's research findings to a single "editor-in-chief"
call that synthesizes them into one ordered, non-redundant article outline. This is the step that turns
N independent research threads into a single coherent narrative structure.

In [ ]:
# ============ OUTLINE SCHEMA & SYNTHESIS ============
class ArticleSection(BaseModel):
    title: str = Field(description="Section heading")
    scope: str = Field(description="One-line description of what this section covers")


class ArticleOutline(BaseModel):
    sections: List[ArticleSection] = Field(description="Ordered sections for the final article")


outline_llm = llm.with_structured_output(ArticleOutline)

research_summary = "\n\n".join(
    f"### {persona}\n" + "\n".join(f"- {fact}" for fact in findings.facts)
    for persona, findings in research_notes.items()
)

outline = outline_llm.invoke(
    [
        SystemMessage(
            content=(
                "You are a senior editor synthesizing multi-perspective research into a coherent "
                "long-form article outline. Produce 4-6 ordered sections that together cover all key "
                "angles below without redundancy."
            )
        ),
        HumanMessage(
            content=f"Topic: {TOPIC}\n\nResearch gathered from expert interviews:\n{research_summary}"
        ),
    ]
)

In [ ]:
# ============ PRINT THE OUTLINE ============
print(f"Article outline for: {TOPIC}\n")
for i, section in enumerate(outline.sections, start=1):
    print(f"{i}. {section.title} — {section.scope}")

**Discussion of the output:** a good outline reads as a single article's table of contents — e.g.
moving from context/background through each stakeholder's stake in the issue to a synthesis or outlook —
rather than "section per persona." If it degenerates into one section per persona, the synthesis prompt
needs to more explicitly ask for a *merged* narrative rather than a concatenation.

## Step 4: Full Article Drafting

**What we are going to do:** draft each outlined section with a writer agent grounded in the combined
research notes, then assemble all sections (plus a title) into the final long-form article.

In [ ]:
# ============ DRAFT EACH SECTION ============
def draft_section(topic: str, section: ArticleSection, research_summary: str) -> str:
    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a long-form journalist writing one section of an article. Write 2-4 "
                    "well-developed paragraphs of prose. Do not repeat the section title as a heading "
                    "in your output."
                )
            ),
            HumanMessage(
                content=(
                    f"Article topic: {topic}\n"
                    f"Section: {section.title}\n"
                    f"Scope: {section.scope}\n\n"
                    f"Relevant research notes from expert interviews:\n{research_summary}\n\n"
                    "Write this section."
                )
            ),
        ]
    )
    return response.content.strip()


section_drafts = [draft_section(TOPIC, section, research_summary) for section in outline.sections]

In [ ]:
# ============ ASSEMBLE THE FINAL ARTICLE ============
title_response = llm.invoke(
    [
        SystemMessage(content="Write a single, punchy article title (no quotes, no trailing period)."),
        HumanMessage(content=f"Topic: {TOPIC}"),
    ]
)
article_title = title_response.content.strip()

body = "\n\n".join(
    f"## {section.title}\n\n{draft}"
    for section, draft in zip(outline.sections, section_drafts)
)
final_article = f"# {article_title}\n\n{body}"


In [ ]:
# ============ PRINT THE FINAL ARTICLE ============
print(final_article)

**Discussion of the output:** each section should visibly draw on the facts surfaced during the
interviews for its relevant persona(s) rather than reading as generic filler — spot-check a sentence or
two per section against `research_notes` to confirm grounding. The whole pipeline (steps 1-4) is a chain
of independently-inspectable agent calls: swapping the Expert agent's plain LLM call for a tool-using
agent (e.g. one bound to `TavilySearch`) would ground the "facts" in real, current sources without
changing anything downstream.

## Summary

STORM is a **multi-agent research-and-writing orchestration pattern**, sibling to Debate and Blackboard
in this folder: instead of agents arguing toward consensus (Debate) or posting to a shared workspace
that any agent may act on (Blackboard), STORM's agents play fixed, cooperative roles in a pipeline —
Perspective Generator, Persona-Editor, Expert, Outline Synthesizer, and Writer — each consuming the
previous stage's structured output.

Key takeaways:

- **Perspective diversity drives research quality.** Generating distinct personas up front is what
  keeps the final article from reading like a single generic take on the topic.
- **Simulated interviews are a cheap substitute for real research** when no retrieval tool is available,
  and a natural place to plug one in (`TavilySearch`) when grounding matters.
- **Structured outputs (Pydantic + `with_structured_output`) are what make the pipeline composable** —
  every stage consumes a typed object produced by the previous one instead of parsing free text.
- **Outline synthesis is the critical hinge step**: it is where N independent research threads become
  one coherent narrative, and it is the step most worth iterating on if the final article feels
  disjointed.
- **Every stage is independently inspectable** (perspectives, transcripts, findings, outline, drafts),
  which makes this pattern easier to debug and trust than a single-shot "write me an article" prompt.